In [1]:
#libaries
import pandas as pd
import numpy as np
import pycountry_convert as pc

In [2]:
df_long = pd.read_csv('../data/outputs/merged_data.csv')

In [6]:
#COVID period flag
def covid_period(year):
  if year < 2020: return 'Pre-COVID'
  elif year == 2020: return 'COVID Peak'
  elif year <= 2022: return 'Recovery'
  else: return 'Post-COVID'

df_long['covid_period'] = df_long['year'].apply(covid_period)
print("covid distribution")
for period, count in df_long['covid_period'].value_counts().sort_index().items():
    print(f"      {period}: {count} ({count/len(df_long)*100:.1f}%)")

covid distribution
      COVID Peak: 740 (9.1%)
      Post-COVID: 1480 (18.2%)
      Pre-COVID: 4440 (54.5%)
      Recovery: 1480 (18.2%)


In [ ]:
# Gender unemployment gap
gender_pivot = df_long.pivot_table(
  index=['country_name', 'year', 'age_categories'],
  columns='sex',
  values='unemployment_rate'
).reset_index()

gender_pivot['gender_gap'] = gender_pivot['Female'] - gender_pivot['Male']
gender_pivot['gender_gap_abs'] = gender_pivot['gender_gap'].abs()
print(f"Gender gap: avg={gender_pivot['gender_gap'].mean():.2f}pp, shape={gender_pivot.shape}")

Gender gap: avg=2.66pp, shape=(4070, 7)


In [ ]:
# Youth-to-adult unemployment ratio
youth = df_long[df_long['age_categories']=='Youth'][['country_name','year','sex','unemployment_rate']]
adult = df_long[df_long['age_categories']=='Adults'][['country_name','year','sex','unemployment_rate']]

ratio_df = youth.merge(adult, on=['country_name','year','sex'], suffixes=('_youth','_adult'))

# Use np.where to handle zero denominators properly (set to NaN instead of artificial +0.01)
ratio_df['youth_adult_ratio'] = np.where(
    ratio_df['unemployment_rate_adult'] == 0,
    np.nan,
    ratio_df['unemployment_rate_youth'] / ratio_df['unemployment_rate_adult']
)

# Validation
print("Youth-adult ratio feature created")
print(f"   Average: {ratio_df['youth_adult_ratio'].mean():.2f}x")
print(f"   Youth > 2x adult: {(ratio_df['youth_adult_ratio']>2).sum()} cases ({(ratio_df['youth_adult_ratio']>2).mean()*100:.0f}%)")
print(f"   Shape: {ratio_df.shape}")

Youth-adult ratio feature created
   Null count: 0
   Average: 3.26x
   Youth > 2x adult: 3417 cases (84%)
   Shape: (4070, 6)


In [ ]:
#Economic development tier
df_long['dev_tier'] = pd.cut(
  df_long['gdp_per_capita_current_usd'],
  bins=[0, 1085, 4255, 13205, float('inf')],
  labels=['Low Income', 'Lower-Middle', 'Upper-Middle', 'High Income']
)
print(f"Dev tier: {df_long['dev_tier'].value_counts().to_dict()}")

Dev tier: {'High Income': 2676, 'Lower-Middle': 2180, 'Upper-Middle': 2124, 'Low Income': 1160}


In [ ]:
# Year-over-year unemployment change
df_long = df_long.sort_values(['country_name','sex','age_categories','year'])
df_long['unemp_yoy_change'] = df_long.groupby(
  ['country_name','sex','age_categories']
)['unemployment_rate'].diff()
df_long['improving'] = (df_long['unemp_yoy_change'] < 0).astype(int)
print(f"YoY change: avg={df_long['unemp_yoy_change'].mean():.2f}pp, improving={df_long['improving'].mean()*100:.0f}%")

YoY change: avg=-0.14pp, improving=51%


In [11]:
# Classify unemployment rate into severity bands
# Based on ILO standards: <5% low, 5-10% moderate, 10-20% high, >20% severe
conditions = [
    df_long['unemployment_rate'] < 5,
    df_long['unemployment_rate'] < 10,
    df_long['unemployment_rate'] < 20,
    df_long['unemployment_rate'] >= 20
]
choices = ['Low', 'Moderate', 'High', 'Severe']
df_long['unemployment_severity'] = np.select(conditions, choices, default='Unknown')

# Validation
print("Unemployment severity feature created")
print(f"Distribution:")
for severity, count in pd.Series(df_long['unemployment_severity']).value_counts().items():
    print(f"      {severity}: {count}")
print(f"\nSeverity by age category:")
severity_by_age = pd.crosstab(df_long['age_categories'], df_long['unemployment_severity'])
print(severity_by_age)

Unemployment severity feature created
Distribution:
      Low: 2711
      Moderate: 1978
      High: 1885
      Severe: 1566

Severity by age category:
unemployment_severity  High   Low  Moderate  Severe
age_categories                                     
Adults                  618  2200      1099     153
Youth                  1267   511       879    1413


In [ ]:
# Save enhanced dataset
df_long.to_csv('../data/outputs/merged_data_features.csv', index=False)
gender_pivot.to_csv('../data/outputs/gender_gap.csv', index=False)
ratio_df.to_csv('../data/outputs/youth_adult_ratio.csv', index=False)